In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.legend_handler import HandlerBase
import common

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os


def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
ratios = [0.05, 0.1, 0.2, 0.5]
lambdas = 0.005
# ratios = [0.2]
# lambdas = 0.01
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
# method = "BarlowTwins_simple_background"
# directory = f"datas/lambda_{method}_{lambdas}"
directory = f"datas/lambda_{lambdas}_simple_color"
for r in ratios:
    loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
    [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
    eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
    loss.append(loss_array)
    acc_object.append(acc_object_array)
    acc_background.append(acc_background_array)
    steps.append(step_array)
    eig_list.append(eigvals_lists_array)

In [ ]:
background_predictability = 70

fig, axs = plt.subplots(1, 2, figsize=(23, 4.2))

plot_range = 150

positions = []
plateau_range = 3
for r in range(len(ratios)):
    pos = np.where(acc_object[r] > background_predictability + plateau_range)
    if len(pos[0]) > 0 and r != 0:
        positions.append(pos[0][0])
        axs[0].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
        axs[1].axvline(steps[r][positions[-1]], linestyle="--", color=colors[r], linewidth=4, alpha=0.7)
    axs[0].plot(steps[r][:plot_range], acc_object[r][:plot_range], alpha=0.6, linewidth=4, color=colors[r],
                label=f"Object ratio : {ratios[r]:.3f}")
    # axs.plot(steps[r], acc_background[r], linestyle="--", alpha=0.75, linewidth=4, color=colors[r*2], label=f"Object ratio : {ratios[r]:.3f}")

upper_y = np.zeros_like(steps[0][:plot_range]) + background_predictability + plateau_range
lower_y = np.zeros_like(steps[0][:plot_range]) + background_predictability - plateau_range
axs[0].fill_between(steps[0][:plot_range], upper_y, lower_y, color='skyblue', alpha=0.4)
axs[0].text(500, 67, "bg correlation")

# axs.set_xlim(0-1, epochs)
# axs.set_ylim(10, 100)
axs[0].axhline(background_predictability, linestyle="--", color="black", linewidth=2, alpha=0.4)
# class AnyObjectHandler(HandlerBase):
#     def create_artists(self, legend, orig_handle,
#                        x0, y0, width, height, fontsize, trans):
#         l1 = plt.Line2D([x0,y0+width], [0.7*height,0.7*height],
#                                                 linestyle='--', color=colors[r*2])
#         l2 = plt.Line2D([x0,y0+width], [0.3*height,0.3*height], color=colors[r*2+1])
#         return [l1, l2]
# plt.legend([object], [f'Object ratio : {ratios[0]:.2f}'],  handler_map={object: AnyObjectHandler()})
plt.legend()
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Test Accuracy")
axs[0].set_ylim(60, 82)
# plt.tight_layout()
# plt.savefig("figure/loss_acc.pdf")

for r in range(len(ratios)):
    plt.plot(loss[r][:plot_range * 10], label=f"ratio: {ratios[r]}", color=colors[r], alpha=0.6, linewidth=3)
    axs[1].set_xlabel("Step")
    axs[1].set_ylabel("Loss")
# plt.ylim(100, 300)
# plt.yscale('log')
axs[0].legend(ncols=2)

plt.tight_layout()
plt.savefig('figure/waterbirds_color_background.pdf', format='pdf')
plt.show()

In [ ]:
for r in range(len(ratios)):
    for i in range(len(eig_list[r])):
        plt.plot(steps[r][:plot_range], eig_list[r][i][:plot_range], linewidth=2.0)
    plt.yscale('log')
    axs.set_xlabel("Step")
    axs.set_ylabel("C Eigenvalue")
    plt.title(f"Object ratio : {ratios[r]:.3f}")
    # plt.ylim(1e-6, 10)
    plt.show()

In [ ]:
for r in range(len(ratios)):
    plt.plot(loss[r][:plot_range * 10], label=f"ratio: {ratios[r]}", color=colors[r * 2 + 1], alpha=0.6, linewidth=3)
    axs.set_xlabel("Step")
    axs.set_ylabel("C Eigenvalue")
# plt.ylim(100, 300)
# plt.yscale('log')
plt.legend()
plt.show()